# Letterboxd → TMDB scrape

Pulls every rated film from the usernames in `usernames.csv`, resolves each Letterboxd film slug to a TMDB id, then fetches genres from TMDB.

**Output schema** (`letterboxd_reviews.csv`):

`username, tmdb_id, movie_title, rating, genre, subgenre_1, subgenre_2, subgenre_3, subgenre_4, subgenre_5`

The pipeline runs in three phases with checkpoint files between each — if anything crashes, just re-run the cells; finished work is loaded from disk.

| Phase | Input | Output | Workers |
|---|---|---|---|
| 1. Scrape user films | `usernames.csv` | `phase1_user_films.jsonl` | per username |
| 2. Resolve slug → TMDB id | unique slugs from phase 1 | `phase2_slug_to_tmdb.jsonl` | per slug |
| 3. Fetch genres from TMDB | unique tmdb_ids from phase 2 | `phase3_tmdb_genres.jsonl` | per tmdb_id |
| 4. Merge → final CSV | all three checkpoints | `letterboxd_reviews.csv` | — |


## 1. Configuration

In [ ]:
# === EDIT THESE ===
TMDB_API_KEY = "PASTE_YOUR_TMDB_V3_API_KEY_HERE"   # https://www.themoviedb.org/settings/api

# Path to the input usernames CSV (single column, header "username")
USERNAMES_CSV = "usernames.csv"

# Where checkpoints + final CSV are written
OUTPUT_DIR = "."

# Limit while testing; set to None to scrape everyone
MAX_USERS = None         # e.g. 25 for a smoke test

# Multiprocessing — auto = os.cpu_count()
import os
WORKERS = os.cpu_count() or 8
print(f"Using {WORKERS} worker processes")


## 2. Imports & worker module

The heavy-lifting functions live in `lbx_workers.py` next to this notebook. Multiprocessing on macOS uses *spawn*, which requires importable (not notebook-defined) functions — that's why they're in a separate file.

In [ ]:
import csv
import json
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import pandas as pd

# make the notebook's directory importable
NB_DIR = Path(OUTPUT_DIR).resolve()
sys.path.insert(0, str(NB_DIR))

import lbx_workers  # noqa: E402  (must come after sys.path tweak)
from lbx_workers import scrape_user_films, resolve_tmdb_id, fetch_tmdb_genres  # noqa: E402

PHASE1_PATH = NB_DIR / "phase1_user_films.jsonl"
PHASE2_PATH = NB_DIR / "phase2_slug_to_tmdb.jsonl"
PHASE3_PATH = NB_DIR / "phase3_tmdb_genres.jsonl"
FINAL_CSV   = NB_DIR / "letterboxd_reviews.csv"

print("Workers module:", lbx_workers.__file__)


## 2.5. Quick connectivity & API-key check

Run this once before kicking off Phase 1 — it scrapes a single user's first page and pings TMDB so you catch issues (bad API key, blocked network, changed selectors) in seconds rather than hours.


In [ ]:
# 1) Scrape one page from one user — should print >0 films and at least some ratings
_films = scrape_user_films('schaffrillas', max_pages=1)
print(f'sample user pulled: {len(_films)} films, '
      f'{sum(1 for f in _films if f["rating"] is not None)} rated')
for f in _films[:3]:
    print(' ', f)

# 2) Resolve a known slug to a TMDB id
_resolved = resolve_tmdb_id('parasite-2019')
print('resolve parasite-2019 ->', _resolved)
assert _resolved['tmdb_id'] == 496243, f'unexpected tmdb id: {_resolved}'

# 3) Hit TMDB once with the API key
_g = fetch_tmdb_genres((496243, TMDB_API_KEY))
print('parasite genres ->', _g)
assert _g['genres'], 'TMDB returned no genres — check your API key'
print('\nAll three checks passed. Proceed to Phase 1.')


## 3. Load usernames

In [ ]:
usernames_df = pd.read_csv(USERNAMES_CSV)
usernames = (
    usernames_df["username"]
    .dropna().astype(str).str.strip().str.lower()
    .unique().tolist()
)
if MAX_USERS:
    usernames = usernames[:MAX_USERS]
print(f"{len(usernames):,} usernames queued")
usernames[:5]


## 4. Phase 1 — scrape every user's `/films/` page

Per worker we keep a `requests.Session`, retry 429s with exponential backoff, and walk pagination until an empty page. Each completed user is appended to `phase1_user_films.jsonl` immediately so progress is durable.

Re-running this cell skips usernames already present in the checkpoint.

In [ ]:
def _load_done_usernames(path: Path) -> set[str]:
    if not path.exists():
        return set()
    done = set()
    with path.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            done.add(rec["username"])
    return done


def run_phase1(usernames: list[str], workers: int):
    done = _load_done_usernames(PHASE1_PATH)
    todo = [u for u in usernames if u not in done]
    print(f"Phase 1: {len(done):,} already scraped, {len(todo):,} to go")
    if not todo:
        return

    t0 = time.time()
    with PHASE1_PATH.open("a") as fh, Pool(processes=workers) as pool:
        for i, films in enumerate(pool.imap_unordered(scrape_user_films, todo, chunksize=1), 1):
            if not films:
                # still record the username so we don't retry empty profiles forever
                fh.write(json.dumps({"username": todo[i-1] if i-1 < len(todo) else "", "films": []}) + "\n")
            else:
                fh.write(json.dumps({"username": films[0]["username"], "films": films}) + "\n")
            fh.flush()
            if i % 25 == 0 or i == len(todo):
                rate = i / (time.time() - t0 + 1e-9)
                print(f"  {i:,}/{len(todo):,} users  ({rate:.1f}/s)")

run_phase1(usernames, WORKERS)


## 5. Aggregate phase 1 → unique slugs

In [ ]:
rows = []
with PHASE1_PATH.open() as fh:
    for line in fh:
        rec = json.loads(line)
        rows.extend(rec.get("films", []))

films_df = pd.DataFrame(rows)
print(f"{len(films_df):,} (user, film) rows across {films_df['username'].nunique():,} users")
unique_slugs = films_df["slug"].dropna().unique().tolist()
print(f"{len(unique_slugs):,} unique film slugs to resolve")
films_df.head()


## 6. Phase 2 — resolve each slug to a TMDB id

This is the expensive part: one HTTP request per unique film page on Letterboxd. With cpu_count workers and a session per worker it should hit a few hundred slugs/second on a fast connection.

In [ ]:
def _load_done_slugs(path: Path) -> set[str]:
    if not path.exists():
        return set()
    done = set()
    with path.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            done.add(rec["slug"])
    return done


def run_phase2(slugs: list[str], workers: int):
    done = _load_done_slugs(PHASE2_PATH)
    todo = [s for s in slugs if s not in done]
    print(f"Phase 2: {len(done):,} resolved, {len(todo):,} to go")
    if not todo:
        return

    t0 = time.time()
    with PHASE2_PATH.open("a") as fh, Pool(processes=workers) as pool:
        for i, rec in enumerate(pool.imap_unordered(resolve_tmdb_id, todo, chunksize=4), 1):
            fh.write(json.dumps(rec) + "\n")
            if i % 200 == 0 or i == len(todo):
                fh.flush()
                rate = i / (time.time() - t0 + 1e-9)
                print(f"  {i:,}/{len(todo):,} slugs  ({rate:.1f}/s)")

run_phase2(unique_slugs, WORKERS)


## 7. Aggregate phase 2 → unique TMDB ids

In [ ]:
slug_rows = []
with PHASE2_PATH.open() as fh:
    for line in fh:
        slug_rows.append(json.loads(line))

slug_df = pd.DataFrame(slug_rows)
slug_df["tmdb_id"] = pd.to_numeric(slug_df["tmdb_id"], errors="coerce").astype("Int64")
print(f"{slug_df['tmdb_id'].notna().sum():,} / {len(slug_df):,} slugs resolved to a TMDB id")
unique_tmdb_ids = slug_df["tmdb_id"].dropna().astype(int).unique().tolist()
print(f"{len(unique_tmdb_ids):,} unique TMDB ids to fetch genres for")


## 8. Phase 3 — fetch genres from TMDB

TMDB's v3 API rate-limit is generous (~50 req/s) — we honor `Retry-After` if we ever see a 429. One row per TMDB id, with the full genre list.

In [ ]:
def _load_done_tmdb(path: Path) -> set[int]:
    if not path.exists():
        return set()
    done = set()
    with path.open() as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            done.add(int(rec["tmdb_id"]))
    return done


def run_phase3(tmdb_ids: list[int], api_key: str, workers: int):
    if not api_key or api_key.startswith("PASTE_"):
        raise RuntimeError("Set TMDB_API_KEY in the config cell first.")
    done = _load_done_tmdb(PHASE3_PATH)
    todo = [t for t in tmdb_ids if t not in done]
    print(f"Phase 3: {len(done):,} fetched, {len(todo):,} to go")
    if not todo:
        return

    args = [(t, api_key) for t in todo]
    t0 = time.time()
    with PHASE3_PATH.open("a") as fh, Pool(processes=workers) as pool:
        for i, rec in enumerate(pool.imap_unordered(fetch_tmdb_genres, args, chunksize=8), 1):
            fh.write(json.dumps(rec) + "\n")
            if i % 500 == 0 or i == len(todo):
                fh.flush()
                rate = i / (time.time() - t0 + 1e-9)
                print(f"  {i:,}/{len(todo):,} ids  ({rate:.1f}/s)")

run_phase3(unique_tmdb_ids, TMDB_API_KEY, WORKERS)


## 9. Build the final CSV

Join films × slug→tmdb × tmdb→genres and explode the genre list into `genre` + `subgenre_1..5`.

In [ ]:
genre_rows = []
with PHASE3_PATH.open() as fh:
    for line in fh:
        genre_rows.append(json.loads(line))
genre_df = pd.DataFrame(genre_rows)
genre_df["tmdb_id"] = genre_df["tmdb_id"].astype("Int64")

# Re-load aggregates in case the kernel was restarted between phases.
films = []
with PHASE1_PATH.open() as fh:
    for line in fh:
        rec = json.loads(line)
        films.extend(rec.get("films", []))
films_df = pd.DataFrame(films)

slug_rows = []
with PHASE2_PATH.open() as fh:
    for line in fh:
        slug_rows.append(json.loads(line))
slug_df = pd.DataFrame(slug_rows)
slug_df["tmdb_id"] = pd.to_numeric(slug_df["tmdb_id"], errors="coerce").astype("Int64")

# join: films -> slug -> tmdb -> genres
merged = (
    films_df.merge(slug_df[["slug", "tmdb_id"]], on="slug", how="left")
            .merge(genre_df[["tmdb_id", "title", "genres"]], on="tmdb_id", how="left",
                   suffixes=("", "_tmdb"))
)
# Prefer TMDB's canonical title when present
merged["movie_title"] = merged["title_tmdb"].fillna(merged["title"])

# Explode genres into 6 columns: genre, subgenre_1..5
def split_genres(gs):
    gs = gs if isinstance(gs, list) else []
    gs = (gs + [None] * 6)[:6]
    return pd.Series(gs, index=["genre", "subgenre_1", "subgenre_2", "subgenre_3", "subgenre_4", "subgenre_5"])

genre_cols = merged["genres"].apply(split_genres)
final = pd.concat(
    [merged[["username", "tmdb_id", "movie_title", "rating"]], genre_cols],
    axis=1,
)

print(f"Final shape: {final.shape}")
final.head()


In [ ]:
final.to_csv(FINAL_CSV, index=False)
size_mb = FINAL_CSV.stat().st_size / 1024 / 1024
print(f"Wrote {FINAL_CSV}  ({len(final):,} rows, {size_mb:.1f} MB)")


## 10. Sanity checks

In [ ]:
print("Rows                :", f"{len(final):,}")
print("Unique users        :", f"{final['username'].nunique():,}")
print("Unique movies (tmdb):", f"{final['tmdb_id'].nunique():,}")
print("With rating         :", f"{final['rating'].notna().sum():,}")
print("With tmdb_id        :", f"{final['tmdb_id'].notna().sum():,}")
print("With at least 1 genre:", f"{final['genre'].notna().sum():,}")
print()
print("Top genres:")
print(final["genre"].value_counts().head(10))
